<!-- STEP A0 -->
## Setup: libraries, folders and the scores we report

This first block prepares everything the rest of the notebook needs: it installs `shap` and `lime` if
they are missing (only on Google Colab), imports the libraries, fixes the random seed (`42`) and defines
the two functions that compute the scores the brief asks for. The data are loaded in Part A below.

In [1]:
# STEP A0
# Install shap and lime only if they are missing (Google Colab). Locally they come from requirements.txt.
import importlib.util
import subprocess
import sys

_missing = [p for p in ("shap", "lime") if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

In [2]:
# STEP A0
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             recall_score)
from sklearn.model_selection import train_test_split

SEED = 42
SETUP_START = time.time()

# scikit-learn 1.6.1 passes an option that newer scipy no longer knows; the warning is harmless.
warnings.filterwarnings("ignore", message="Unknown solver options")

# The part notebooks live in parts/, the hand-in notebook in the project root. Work from the root in
# both cases, so that UNSW-NB15/ and results/ mean the same thing everywhere.
if Path.cwd().name == "parts":
    os.chdir("..")

# The data: UNSW-NB15/ in the project, or the CSV files uploaded next to the notebook (Google Colab).
DATA_DIR = Path("UNSW-NB15")
if not (DATA_DIR / "Data.csv").is_file() and Path("Data.csv").is_file():
    DATA_DIR = Path(".")

TABLES = Path("results/tables")
FIGURES = Path("results/figures")
for folder in (TABLES, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Python {sys.version.split()[0]}, scikit-learn {sklearn.__version__}, "
      f"numpy {np.__version__}, pandas {pd.__version__}")
print(f"Working directory: {Path.cwd()}")
print(f"Data folder: {DATA_DIR.resolve()}")

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15


<!-- STEP A0 -->
### `score()`: the scores the brief asks for

The brief asks for **macro-F1, recall, PR-AUC and FAR** for every model. We also report accuracy,
because the trivial baseline (always "benign") is defined by it.

- **Recall** = the share of real attacks the model catches.
- **FAR** (false alarm rate) = FP / (FP + TN): the share of benign flows wrongly called attacks.
- **PR-AUC** = average precision: how well the model ranks attacks above benign flows. A model that
  guesses scores the attack share.
- **Macro-F1** = the F1 of the attack class and the F1 of the benign class, averaged equally.

`score(model, X, y)` works for any model with `predict_proba`. `score_proba(y, p)` takes attack
probabilities directly; Part F uses it for the belief rule base, which is not a scikit-learn model.

In [3]:
# STEP A0
def score_proba(y, p, threshold=0.5):
    """The five scores from attack probabilities p (a flow is called an attack if p >= threshold)."""
    p = np.asarray(p, dtype=float)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "recall": recall_score(y, pred, zero_division=0),
        "pr_auc": average_precision_score(y, p),
        "far": float(fp / (fp + tn)) if (fp + tn) else 0.0,
    }


def score(model, X, y, threshold=0.5):
    """Score any model with predict_proba on (X, y)."""
    return score_proba(y, model.predict_proba(X)[:, 1], threshold)


# Quick check on a tiny hand-made example: 2 attacks (1 caught), 2 benign (1 false alarm).
_s = score_proba([1, 1, 0, 0], [0.9, 0.2, 0.7, 0.1])
assert _s["recall"] == 0.5 and _s["far"] == 0.5 and _s["accuracy"] == 0.5
print("score() and score_proba() ready")

score() and score_proba() ready


<!-- STEP A2 -->
## Part A: load and clean the data

**Dataset.** CIC-UNSW-NB15: the UNSW-NB15 traffic (recorded in 2015 at UNSW Canberra) turned into
network flows by CICFlowMeter, from the Canadian Institute for Cybersecurity. One row is one flow (one
conversation between two computers) described by 76 numbers. `Data.csv` holds the features and
`Label.csv` the label of each row: 0 = benign, 1–9 = one of nine attack types (`Readme.txt`).

We use `Data.csv`, not the much bigger `CICFlowMeter_out.csv`: that file also holds the IP addresses,
and every attack comes from the same few attacker addresses, so a model could "cheat" by reading them.
`Data.csv` keeps all attacks but only about 10% of the benign flows, so 20% of its rows are attacks,
against about 2.5% in the full recording.

**Cleaning**, in this order:
1. **Exact duplicates** are removed. If one copy of a flow were in training and another in the test
   set, the test would be too easy.
2. **Conflicting rows** are removed: flows whose 76 numbers are identical but whose labels differ. No
   model can get them right.
3. Columns with a **single constant value** in the training set are removed after the split (Part A3):
   they carry no information.

We keep **all** remaining rows; there is no subsample.

In [4]:
# STEP A2
X_raw = pd.read_csv(DATA_DIR / "Data.csv")
labels = pd.read_csv(DATA_DIR / "Label.csv")["Label"]
# Keep all 76 columns as decimal numbers in one block of memory: selecting rows (the split) is then
# about 3x faster. The integer columns are converted exactly (their largest value is below 10^9).
X_raw = pd.DataFrame(X_raw.to_numpy(dtype=float), columns=X_raw.columns)
assert len(X_raw) == len(labels) == 447_915, (len(X_raw), len(labels))
assert X_raw.shape[1] == 76, X_raw.shape

# Label codes from Readme.txt, e.g. "Fuzzers  5". The file is tab-separated with varying tabs.
ATTACK_NAMES = {}
for line in (DATA_DIR / "Readme.txt").read_text().splitlines():
    parts = line.split()
    if len(parts) == 2 and parts[1].isdigit():
        ATTACK_NAMES[int(parts[1])] = parts[0]
assert ATTACK_NAMES == {0: "Benign", 1: "Analysis", 2: "Backdoor", 3: "DoS", 4: "Exploits",
                        5: "Fuzzers", 6: "Generic", 7: "Reconnaissance", 8: "Shellcode", 9: "Worms"}

feature_columns = list(X_raw.columns)
attack_type = labels.map(ATTACK_NAMES)
assert attack_type.notna().all(), "a label code is not in Readme.txt"

# No infinite or missing values allowed (checked: there are none).
finite = np.isfinite(X_raw.to_numpy()).all(axis=1)
print(f"Rows with an infinite or missing value: {int((~finite).sum())}")
df = X_raw[finite].assign(attack_type=attack_type[finite])
del X_raw, labels, attack_type
print(f"Loaded {len(df):,} flows x {len(feature_columns)} features")

Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


In [5]:
# STEP A2
raw_counts = df["attack_type"].value_counts()
n_raw = len(df)

# 1. Exact duplicates (same 76 numbers and same label). Hashing each row and keeping the first copy
#    gives exactly the rows df.drop_duplicates() keeps (checked), in 4 s instead of 21 s.
df = df[~pd.util.hash_pandas_object(df, index=False).duplicated().to_numpy()]
n_after_duplicates = len(df)

# 2. Conflicting rows: identical 76 numbers, different labels. A hash of each row's numbers is a fast
#    way to find identical rows.
row_key = pd.util.hash_pandas_object(df[feature_columns], index=False).to_numpy()
conflict = df.groupby(row_key)["attack_type"].transform("nunique").to_numpy() > 1
df = df[~conflict]
n_after_conflicts = len(df)

CLEAN_COUNTS = {"raw": n_raw, "after_duplicates": n_after_duplicates,
                "after_conflicts": n_after_conflicts}
assert CLEAN_COUNTS == {"raw": 447_915, "after_duplicates": 306_173, "after_conflicts": 305_105}, CLEAN_COUNTS
print(f"Raw rows:                    {n_raw:>9,}")
print(f"Exact duplicates removed:    {n_raw - n_after_duplicates:>9,}")
print(f"Conflicting rows removed:    {n_after_duplicates - n_after_conflicts:>9,}")
print(f"Clean rows (all are used):   {n_after_conflicts:>9,}")

Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


In [6]:
# STEP A2
# Class balance before and after cleaning.
clean_counts = df["attack_type"].value_counts()
order = ["Benign"] + [n for n in clean_counts.index if n != "Benign"]
class_balance = pd.DataFrame({
    "raw_rows": raw_counts.reindex(order),
    "clean_rows": clean_counts.reindex(order),
})
class_balance["clean_share"] = class_balance["clean_rows"] / class_balance["clean_rows"].sum()
class_balance.loc["All attacks"] = [
    class_balance.drop(index="Benign")["raw_rows"].sum(),
    class_balance.drop(index="Benign")["clean_rows"].sum(),
    1 - class_balance.loc["Benign", "clean_share"],
]
class_balance.index.name = "attack_type"
class_balance.to_csv(TABLES / "A2_class_balance.csv", float_format="%.4f")
class_balance.style.format({"raw_rows": "{:,.0f}", "clean_rows": "{:,.0f}", "clean_share": "{:.2%}"})

,raw_rows,clean_rows,clean_share
attack_type,,,
Benign,"358,332","224,847",73.69%
Exploits,"30,951","30,836",10.11%
Fuzzers,"29,613","26,010",8.52%
Reconnaissance,"16,735","11,988",3.93%
Generic,"4,632","4,373",1.43%
DoS,"4,467","4,315",1.41%
Shellcode,"2,102","1,663",0.55%
Backdoor,452,452,0.15%
Analysis,385,385,0.13%


<!-- STEP A3 -->
## Part A: split 60 / 20 / 20

The clean rows are split into **training** (60%: the models learn from it), **validation** (20%: used
to choose every setting) and **test** (20%: only for the final scores). The split is **stratified on
the attack type**, so every attack type has the same share in all three parts, and uses seed 42.
First 20% is cut off as the test set, then 25% of the remaining 80% becomes the validation set.

Then, measured on the **training set only** (measuring on validation or test data would leak
information):
- columns that hold one constant value are dropped from all three parts, which leaves 67 features;
- `FLAG_COLS` are the 0/1 columns, `CONT_COLS` the continuous ones (Part E adds noise only to these);
- `TRAIN_MEDIAN` and `TRAIN_STD` are each feature's median and standard deviation (Parts D and E);
- `TUNE_IDX` is a stratified 20% part of the training set, used only to try settings quickly in the
  settings searches (Parts B, F). The chosen model is then trained on the whole training set.

In [7]:
# STEP A3
y = (df["attack_type"] != "Benign").astype(int)          # 1 = attack, 0 = benign
X_all = df[feature_columns]

# Split the row positions (fast), then take the rows once. Same result as splitting the tables.
types = df["attack_type"]
pos_rest, pos_test = train_test_split(np.arange(len(df)), test_size=0.20, stratify=types,
                                      random_state=SEED)
pos_train, pos_val = train_test_split(pos_rest, test_size=0.25, stratify=types.iloc[pos_rest],
                                      random_state=SEED)
X_train, X_val, X_test = (X_all.iloc[p] for p in (pos_train, pos_val, pos_test))
y_train, y_val, y_test = (y.iloc[p] for p in (pos_train, pos_val, pos_test))
type_train, type_val, type_test = (types.iloc[p] for p in (pos_train, pos_val, pos_test))
del X_all, pos_rest

assert (len(X_train), len(X_val), len(X_test)) == (183_063, 61_021, 61_021)

# Columns with one constant value in the TRAINING set carry no information: drop them everywhere.
# 0/1 columns: exactly two values, i.e. every value is the column's minimum or its maximum.
_v = X_train.to_numpy()
_low, _high = _v.min(axis=0), _v.max(axis=0)
_is_constant = _low == _high
_is_flag = ~_is_constant & ((_v == _low) | (_v == _high)).all(axis=0)
constant = [c for c, k in zip(feature_columns, _is_constant) if k]
FEATURES = [c for c in feature_columns if c not in constant]
FLAG_COLS = [c for c, k in zip(feature_columns, _is_flag) if k]
CONT_COLS = [c for c in FEATURES if c not in FLAG_COLS]
X_train, X_val, X_test = X_train[FEATURES], X_val[FEATURES], X_test[FEATURES]
del _v
assert len(FEATURES) == 67, len(FEATURES)
TRAIN_MEDIAN = X_train.median()
TRAIN_STD = X_train.std()

# Positions of a stratified 20% part of the training set, only for quick settings searches.
TUNE_IDX = np.sort(train_test_split(np.arange(len(X_train)), train_size=0.20, stratify=type_train,
                                    random_state=SEED)[0])

print(f"Dropped {len(constant)} constant columns: {', '.join(constant)}")
print(f"{len(FEATURES)} features: {len(CONT_COLS)} continuous, {len(FLAG_COLS)} 0/1 columns "
      f"({', '.join(FLAG_COLS)})")
print(f"TUNE_IDX: {len(TUNE_IDX):,} training rows for settings searches")

Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches


In [8]:
# STEP A3
# Rows per part, attack share per part, and rows of each attack type per part.
split = pd.DataFrame({"train": type_train.value_counts(), "validation": type_val.value_counts(),
                      "test": type_test.value_counts()}).reindex(order)
split.loc["All rows"] = split.sum()
split.loc["Attack share"] = [y_train.mean(), y_val.mean(), y_test.mean()]
split.index.name = "attack_type"
split.to_csv(TABLES / "A3_split.csv", float_format="%.4f")

assert split.loc["Fuzzers"].tolist() == [15_606, 5_202, 5_202]
assert int(y_test.sum()) == 16_052
split.style.format(lambda v: f"{v:.2%}" if v < 1 else f"{v:,.0f}")

,train,validation,test
attack_type,,,
Benign,"134,909","44,969","44,969"
Exploits,"18,502","6,167","6,167"
Fuzzers,"15,606","5,202","5,202"
Reconnaissance,"7,192","2,398","2,398"
Generic,"2,623",875,875
DoS,"2,589",863,863
Shellcode,998,332,333
Backdoor,271,91,90
Analysis,231,77,77


In [9]:
# STEP A3
# Everything the later parts rely on (TASKLIST.md, section 2.3). Fail here, not halfway through Part G.
_shared = ["SEED", "ATTACK_NAMES", "CLEAN_COUNTS", "X_train", "X_val", "X_test", "y_train", "y_val",
           "y_test", "type_train", "type_val", "type_test", "TUNE_IDX", "FEATURES", "FLAG_COLS",
           "CONT_COLS", "TRAIN_MEDIAN", "TRAIN_STD", "score", "score_proba", "TABLES", "FIGURES"]
_missing_names = [name for name in _shared if name not in globals()]
assert not _missing_names, f"setup did not create: {_missing_names}"
for _part in (X_train, X_val, X_test):
    assert list(_part.columns) == FEATURES
del df, row_key, conflict, finite, types
print(f"Setup complete: {len(_shared)} shared names ready in {time.time() - SETUP_START:.0f} s")

Setup complete: 22 shared names ready in 46 s
